In [ ]:
"""
CDC WONDER — Suicide Deaths by State

Pulls state-level suicide death counts from CDC WONDER's "Underlying Cause of
Death" database using WONDER's POST/XML API (no key required).

Suicide is defined per NCHS's own standard: ICD-10 underlying-cause-of-death
codes U03, X60-X84, Y87.0. Source: CDC NCHS Data Briefs (e.g. #309, #433, #541).

Longest available *consistent* timeline: 1999-2020 (database D76, final,
bridged-race) extended with 2021-2024 (database D158, final, single-race).
ICD-10 only goes back to 1999 — earlier years used ICD-9/ICD-8 with different
codes, so they aren't combined here. See notes at the bottom of this file.
"""

import time
import requests
import pandas as pd
import bs4 as bs

# --- Suicide definition (NCHS standard) -------------------------------------
SUICIDE_ICD10_CODES = ["U03"] + [f"X{n}" for n in range(60, 85)] + ["Y870"]  # Y87.0

WONDER_DATABASES = {
    "D76":  {"label": "Underlying Cause of Death, 1999-2020 (Bridged-Race)",
             "years": [str(y) for y in range(1999, 2021)],
             "verified": True},
    "D158": {"label": "Underlying Cause of Death, 2018-2024 (Single-Race)",
             "years": [str(y) for y in range(2021, 2025)],  # 2021-24 only, to avoid double-counting 2018-20
             "verified": False},
}


def _param_block(d: dict) -> str:
    out = ""
    for key, val in d.items():
        out += f"<parameter>\n<name>{key}</name>\n"
        for v in (val if isinstance(val, list) else [val]):
            out += f"<value>{v}</value>\n"
        out += "</parameter>\n"
    return out


def _build_xml(db: str, years: list) -> str:
    """WONDER request XML: group by State + Year, filter to suicide ICD-10 codes."""

    b_parameters = {
        "B_1": f"{db}.V9",         # Group by State
        "B_2": f"{db}.V1-level1",  # Group by Year
        "B_3": "*None*", "B_4": "*None*", "B_5": "*None*",
    }
    m_parameters = {
        "M_1": f"{db}.M1",  # Deaths
        "M_2": f"{db}.M2",  # Population
        "M_3": f"{db}.M3",  # Crude Rate
    }
    f_parameters = {
        f"F_{db}.V1": years,
        f"F_{db}.V10": ["*All*"],            # Census Regions
        f"F_{db}.V2": SUICIDE_ICD10_CODES,   # ICD-10 cause filter
        f"F_{db}.V27": ["*All*"],            # HHS Regions
        f"F_{db}.V9": ["*All*"],             # State/County
    }
    i_parameters = {
        f"I_{db}.V1": "*All* (All Dates)",
        f"I_{db}.V10": "*All* (The United States)",
        f"I_{db}.V2": "Suicide (U03, X60-X84, Y87.0)",
        f"I_{db}.V27": "*All* (The United States)",
        f"I_{db}.V9": "*All* (The United States)",
    }
    v_parameters = {
        f"V_{db}.V1": "", f"V_{db}.V10": "", f"V_{db}.V11": "*All*",
        f"V_{db}.V12": "*All*", f"V_{db}.V17": "*All*", f"V_{db}.V19": "*All*",
        f"V_{db}.V2": "", f"V_{db}.V20": "*All*", f"V_{db}.V21": "*All*",
        f"V_{db}.V22": "*All*", f"V_{db}.V23": "*All*", f"V_{db}.V24": "*All*",
        f"V_{db}.V25": "*All*", f"V_{db}.V27": "", f"V_{db}.V4": "*All*",
        f"V_{db}.V5": "*All*", f"V_{db}.V51": "*All*", f"V_{db}.V52": "*All*",
        f"V_{db}.V6": "00", f"V_{db}.V7": "*All*", f"V_{db}.V8": "*All*",
        f"V_{db}.V9": "",
    }
    o_parameters = {
        "O_V10_fmode": "freg", "O_V1_fmode": "freg", "O_V27_fmode": "freg",
        "O_V2_fmode": "freg", "O_V9_fmode": "freg",
        "O_aar": "aar_none",          # no age-adjusted rate, keeps output to 5 cols
        "O_javascript": "on",
        "O_location": f"{db}.V9",
        "O_precision": "1",
        "O_rate_per": "100000",
        "O_show_totals": "false",
        "O_timeout": "300",
        "O_title": "Suicide Deaths by State and Year",
        "O_ucd": f"{db}.V2",
    }
    vm_parameters = {
        f"VM_{db}.M6_{db}.V10": "", f"VM_{db}.M6_{db}.V17": "*All*",
        f"VM_{db}.M6_{db}.V1_S": "*All*", f"VM_{db}.M6_{db}.V7": "*All*",
        f"VM_{db}.M6_{db}.V8": "*All*",
    }
    misc_parameters = {
        "action-Send": "Send",
        f"finder-stage-{db}.V1": "codeset",
        f"finder-stage-{db}.V2": "codeset",
        f"finder-stage-{db}.V27": "codeset",
        f"finder-stage-{db}.V9": "codeset",
        "stage": "request",
    }

    xml = "<request-parameters>\n"
    for block in (b_parameters, m_parameters, f_parameters, i_parameters,
                  o_parameters, vm_parameters, v_parameters, misc_parameters):
        xml += _param_block(block)
    xml += "</request-parameters>"
    return xml


def _parse_response(xml_text: str) -> pd.DataFrame:
    """Parses WONDER's <r>/<c> row/column XML into a DataFrame."""
    root = bs.BeautifulSoup(xml_text, "lxml")
    rows = []
    for row_num, row in enumerate(root.find_all("r")):
        if row_num >= len(rows):
            rows.append([])
        for cell in row.find_all("c"):
            if "v" in cell.attrs:
                raw = cell.attrs["v"].replace(",", "")
                try:
                    raw = float(raw)
                except ValueError:
                    pass  # e.g. "Suppressed" for counts of 9 or fewer
                rows[row_num].append(raw)
            elif "r" not in cell.attrs:
                rows[row_num].append(cell.attrs["l"])
            else:
                for i in range(int(cell.attrs["r"])):
                    if (row_num + i) >= len(rows):
                        rows.append([])
                    rows[row_num + i].append(cell.attrs["l"])

    expected_cols = ["State", "Year", "Deaths", "Population", "Crude Rate"]
    n_cols = len(rows[0]) if rows else len(expected_cols)
    columns = expected_cols[:n_cols] if n_cols <= len(expected_cols) else (
        expected_cols + [f"extra_col_{i}" for i in range(n_cols - len(expected_cols))]
    )
    return pd.DataFrame(rows, columns=columns)


def get_suicide_deaths_by_state(database_id: str, years: list = None,
                                  timeout: int = 300) -> pd.DataFrame:
    """
    Query one WONDER database for suicide deaths grouped by State and Year.

    database_id : e.g. "D76" or "D158" (see WONDER_DATABASES)
    years       : list of year strings, defaults to that database's full range
    """
    years = years or WONDER_DATABASES[database_id]["years"]
    url = f"https://wonder.cdc.gov/controller/datarequest/{database_id}"
    xml_request = _build_xml(database_id, years)

    resp = requests.post(
        url,
        data={"request_xml": xml_request, "accept_datause_restrictions": "true"},
        timeout=timeout,
    )
    resp.raise_for_status()

    if "<error>" in resp.text.lower()[:2000]:
        raise RuntimeError(f"WONDER returned an error for {database_id}:\n{resp.text[:1500]}")

    df = _parse_response(resp.text)
    df["source_db"] = database_id
    return df


def get_longest_available_timeline(out_csv: str = "suicide_deaths_by_state.csv") -> pd.DataFrame:
    """Combines D76 (1999-2020) + D158 (2021-2024) into one state x year CSV."""
    frames = []
    for db, meta in WONDER_DATABASES.items():
        tag = "verified" if meta["verified"] else "UNVERIFIED, best-effort"
        print(f"Querying {db} — {meta['label']} [{tag}]...")
        frames.append(get_suicide_deaths_by_state(db))
        time.sleep(2)  # be polite to WONDER's server between requests

    combined = pd.concat(frames, ignore_index=True).sort_values(["State", "Year"])
    combined.to_csv(out_csv, index=False)
    print(f"Saved {len(combined)} rows to {out_csv}")
    return combined


if __name__ == "__main__":
    df = get_longest_available_timeline()
    print(df.head(20))

# -----------------------------------------------------------------------------
# NOTES / CAVEATS — read before relying on this:
#
# 1. VERIFIED vs UNVERIFIED: The D76 (1999-2020) request structure is adapted
#    directly from a working, tested example (alipphardt/cdc-wonder-api on
#    GitHub) that uses this exact database. The D158 (2021-2024) call reuses
#    the same parameter *pattern*, on the assumption WONDER kept the same
#    variable numbering (V1=year, V2=cause, V9=state...) for that database.
#    I could not confirm this against a working D158 example, and I could not
#    test either call live — wonder.cdc.gov isn't reachable from my sandboxed
#    execution environment. Run get_suicide_deaths_by_state("D158") on its own
#    first and inspect the output before trusting the combined file.
#
# 2. If a call errors or returns garbage: the most likely failure point is
#    "Y870" for Y87.0 — WONDER sometimes wants 4-character ICD-10 codes
#    formatted differently per database. If that code throws an error, drop
#    it from SUICIDE_ICD10_CODES and re-run (Y87.0 is a small share of cases),
#    or pull the working code list from the site's own "XML request" export
#    (run a query manually at https://wonder.cdc.gov/ucd-icd10.html, then
#    download the "Export" -> request XML from the results page).
#
# 3. Small-number suppression: WONDER suppresses death counts of 9 or fewer
#    as the literal string "Suppressed" rather than a number, per federal
#    privacy rules. Those rows will be non-numeric in the Deaths column.
#
# 4. "Longest timeline" beyond 1999: suicide can technically be tracked back
#    to 1968 using ICD-8/ICD-9 codes (E950-E959) via WONDER's "Compressed
#    Mortality" databases, but NCHS doesn't recommend treating ICD-9/ICD-10
#    suicide counts as one continuous series — coding rules changed enough
#    between revisions that trend comparisons across 1998/1999 are considered
#    unreliable. I left that out by default; say the word if you want it
#    added anyway (it needs a separate database ID + parameter set I haven't
#    pulled yet).
#
# 5. Rate limiting: WONDER has no documented API rate limit, but hammering it
#    with rapid requests can get your IP temporarily blocked. The script
#    already sleeps 2s between the two calls — don't parallelize this.
# -----------------------------------------------------------------------------